# Apache Spark com Delta Lake

Demonstração de operações CRUD (INSERT, UPDATE, DELETE) com **PySpark** e **Delta Lake**.

**Cenário:** Sistema de Gestão de Vendas — TechStore  
**Tabelas:** clientes, produtos, pedidos

In [5]:
from pyspark.sql import SparkSession
from delta import *
import logging

logging.getLogger("py4j").setLevel(logging.WARNING)

In [6]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## Cenário — TechStore

Sistema de gestão de vendas de uma loja de eletrônicos com três entidades.

### Modelo ER
```
CLIENTES (1) ----< PEDIDOS >---- (N) PRODUTOS
```
- Um cliente pode realizar vários pedidos
- Um produto pode estar em vários pedidos

## DDL — Criação das Tabelas Delta

In [7]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS clientes (
        id       INT,
        nome     STRING,
        email    STRING,
        cidade   STRING,
        estado   STRING
    )
    USING delta
""")
spark.sql("SELECT * FROM clientes").show()

26/04/23 21:56:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---+----+-----+------+------+
| id|nome|email|cidade|estado|
+---+----+-----+------+------+
+---+----+-----+------+------+



In [8]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS produtos (
        id        INT,
        nome      STRING,
        categoria STRING,
        preco     FLOAT,
        estoque   INT
    )
    USING delta
""")
spark.sql("SELECT * FROM produtos").show()

[Stage 14:========================================>               (36 + 8) / 50]

+---+----+---------+-----+-------+
| id|nome|categoria|preco|estoque|
+---+----+---------+-----+-------+
+---+----+---------+-----+-------+



In [9]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS pedidos (
        id           INT,
        cliente_id   INT,
        produto_id   INT,
        quantidade   INT,
        data_pedido  STRING,
        status       STRING
    )
    USING delta
""")
spark.sql("SELECT * FROM pedidos").show()

+---+----------+----------+----------+-----------+------+
| id|cliente_id|produto_id|quantidade|data_pedido|status|
+---+----------+----------+----------+-----------+------+
+---+----------+----------+----------+-----------+------+



## INSERT — Inserindo Dados

In [10]:
spark.sql("""
    INSERT INTO clientes VALUES
        (1, 'Ana Silva',       'ana@email.com',      'Sao Paulo',      'SP'),
        (2, 'Carlos Oliveira', 'carlos@email.com',   'Rio de Janeiro', 'RJ'),
        (3, 'Maria Santos',    'maria@email.com',    'Curitiba',       'PR'),
        (4, 'Joao Costa',      'joao@email.com',     'Porto Alegre',   'RS'),
        (5, 'Fernanda Lima',   'fernanda@email.com', 'Belo Horizonte', 'MG')
""")
spark.sql("SELECT * FROM clientes").show()

+---+---------------+------------------+--------------+------+
| id|           nome|             email|        cidade|estado|
+---+---------------+------------------+--------------+------+
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|
+---+---------------+------------------+--------------+------+



In [11]:
spark.sql("""
    INSERT INTO produtos VALUES
        (1, 'Notebook Dell',      'Informatica',  3599.99, 15),
        (2, 'Smartphone Samsung', 'Celulares',    1299.00, 50),
        (3, 'Monitor LG 27',      'Informatica',   899.90, 30),
        (4, 'Teclado Mecanico',   'Perifericos',   349.90, 100),
        (5, 'Mouse Logitech',     'Perifericos',   159.90, 80)
""")
spark.sql("SELECT * FROM produtos").show()

+---+------------------+-----------+-------+-------+
| id|              nome|  categoria|  preco|estoque|
+---+------------------+-----------+-------+-------+
|  2|Smartphone Samsung|  Celulares| 1299.0|     50|
|  4|  Teclado Mecanico|Perifericos|  349.9|    100|
|  5|    Mouse Logitech|Perifericos|  159.9|     80|
|  3|     Monitor LG 27|Informatica|  899.9|     30|
|  1|     Notebook Dell|Informatica|3599.99|     15|
+---+------------------+-----------+-------+-------+



In [12]:
spark.sql("""
    INSERT INTO pedidos VALUES
        (1, 1, 2, 2, '2024-01-10', 'entregue'),
        (2, 2, 1, 1, '2024-01-12', 'entregue'),
        (3, 3, 3, 1, '2024-01-15', 'em_transporte'),
        (4, 1, 4, 1, '2024-01-20', 'processando'),
        (5, 4, 5, 3, '2024-01-22', 'cancelado')
""")
spark.sql("SELECT * FROM pedidos").show()

+---+----------+----------+----------+-----------+-------------+
| id|cliente_id|produto_id|quantidade|data_pedido|       status|
+---+----------+----------+----------+-----------+-------------+
|  3|         3|         3|         1| 2024-01-15|em_transporte|
|  4|         1|         4|         1| 2024-01-20|  processando|
|  5|         4|         5|         3| 2024-01-22|    cancelado|
|  2|         2|         1|         1| 2024-01-12|     entregue|
|  1|         1|         2|         2| 2024-01-10|     entregue|
+---+----------+----------+----------+-----------+-------------+



## Consulta com JOIN

In [13]:
spark.sql("""
    SELECT
        p.id         AS pedido_id,
        c.nome       AS cliente,
        pr.nome      AS produto,
        p.quantidade,
        p.status,
        p.data_pedido
    FROM pedidos p
    JOIN clientes c  ON p.cliente_id = c.id
    JOIN produtos pr ON p.produto_id = pr.id
    ORDER BY p.id
""").show(truncate=False)

+---------+---------------+------------------+----------+-------------+-----------+
|pedido_id|cliente        |produto           |quantidade|status       |data_pedido|
+---------+---------------+------------------+----------+-------------+-----------+
|1        |Ana Silva      |Smartphone Samsung|2         |entregue     |2024-01-10 |
|2        |Carlos Oliveira|Notebook Dell     |1         |entregue     |2024-01-12 |
|3        |Maria Santos   |Monitor LG 27     |1         |em_transporte|2024-01-15 |
|4        |Ana Silva      |Teclado Mecanico  |1         |processando  |2024-01-20 |
|5        |Joao Costa     |Mouse Logitech    |3         |cancelado    |2024-01-22 |
+---------+---------------+------------------+----------+-------------+-----------+



## UPDATE — Atualizando Dados

In [14]:
# Atualiza status do pedido 3 para entregue
spark.sql("UPDATE pedidos SET status = 'entregue' WHERE id = 3")
spark.sql("SELECT * FROM pedidos WHERE id = 3").show()

+---+----------+----------+----------+-----------+--------+
| id|cliente_id|produto_id|quantidade|data_pedido|  status|
+---+----------+----------+----------+-----------+--------+
|  3|         3|         3|         1| 2024-01-15|entregue|
+---+----------+----------+----------+-----------+--------+



In [15]:
# Ajusta preco e estoque do produto 2
spark.sql("UPDATE produtos SET preco = 1199.00, estoque = 45 WHERE id = 2")
spark.sql("SELECT * FROM produtos WHERE id = 2").show()

+---+------------------+---------+------+-------+
| id|              nome|categoria| preco|estoque|
+---+------------------+---------+------+-------+
|  2|Smartphone Samsung|Celulares|1199.0|     45|
+---+------------------+---------+------+-------+



## DELETE — Removendo Dados

In [16]:
# Remove pedidos cancelados
spark.sql("DELETE FROM pedidos WHERE status = 'cancelado'")
spark.sql("SELECT * FROM pedidos").show()

+---+----------+----------+----------+-----------+-----------+
| id|cliente_id|produto_id|quantidade|data_pedido|     status|
+---+----------+----------+----------+-----------+-----------+
|  4|         1|         4|         1| 2024-01-20|processando|
|  3|         3|         3|         1| 2024-01-15|   entregue|
|  2|         2|         1|         1| 2024-01-12|   entregue|
|  1|         1|         2|         2| 2024-01-10|   entregue|
+---+----------+----------+----------+-----------+-----------+



## ALTER TABLE — Evolução de Schema

O Delta Lake suporta adicionar colunas sem recriar a tabela.

In [17]:
spark.sql("ALTER TABLE clientes ADD COLUMN telefone STRING")
spark.sql("SELECT * FROM clientes").show()

+---+---------------+------------------+--------------+------+--------+
| id|           nome|             email|        cidade|estado|telefone|
+---+---------------+------------------+--------------+------+--------+
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|    NULL|
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|    NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|    NULL|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|    NULL|
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|    NULL|
+---+---------------+------------------+--------------+------+--------+



In [18]:
spark.sql("UPDATE clientes SET telefone = '(11) 91234-5678' WHERE id = 1")
spark.sql("UPDATE clientes SET telefone = '(21) 99876-5432' WHERE id = 2")
spark.sql("SELECT * FROM clientes").show()

+---+---------------+------------------+--------------+------+---------------+
| id|           nome|             email|        cidade|estado|       telefone|
+---+---------------+------------------+--------------+------+---------------+
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|(21) 99876-5432|
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|(11) 91234-5678|
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|           NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|           NULL|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|           NULL|
+---+---------------+------------------+--------------+------+---------------+



## MERGE — Upsert (Insert or Update)

Insere o registro se não existir, ou atualiza se já existir.

In [19]:
spark.sql("""
    MERGE INTO clientes AS target
    USING (
        SELECT 6 AS id, 'Pedro Alves' AS nome, 'pedro@email.com' AS email,
               'Fortaleza' AS cidade, 'CE' AS estado, '(85) 98765-4321' AS telefone
    ) AS source
    ON target.id = source.id
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")
spark.sql("SELECT * FROM clientes").show()

+---+---------------+------------------+--------------+------+---------------+
| id|           nome|             email|        cidade|estado|       telefone|
+---+---------------+------------------+--------------+------+---------------+
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|(21) 99876-5432|
|  6|    Pedro Alves|   pedro@email.com|     Fortaleza|    CE|(85) 98765-4321|
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|(11) 91234-5678|
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|           NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|           NULL|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|           NULL|
+---+---------------+------------------+--------------+------+---------------+



## Time Travel — Viagem no Tempo

O Delta Lake mantém um **transaction log** que permite consultar versões anteriores.

In [20]:
# Historico completo de transacoes da tabela clientes
spark.sql("DESCRIBE HISTORY clientes").show(truncate=False)

+-------+-----------------------+------+--------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp 

In [21]:
from delta.tables import DeltaTable

# Verifica se e tabela Delta
print(DeltaTable.isDeltaTable(spark, "spark-warehouse/clientes"))

# Le versao 0 (estado inicial - apenas INSERT)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("spark-warehouse/clientes")
print("Clientes na versao 0:")
df_v0.show()

False


Py4JJavaError: An error occurred while calling o124.load.
: java.io.FileNotFoundException: No such file or directory: file:/mnt/c/Users/mazuc/OneDrive/Área de Trabalho/SATC/5° FASE/Engenharia de Dados/Trabalho Apache spark iceberg e Data Lake/notebooks/spark-warehouse/clientes/_delta_log
	at io.delta.storage.HadoopFileSystemLogStore.listFrom(HadoopFileSystemLogStore.java:56)
	at org.apache.spark.sql.delta.storage.LogStoreAdaptor.listFrom(LogStore.scala:452)
	at org.apache.spark.sql.delta.storage.DelegatingLogStore.listFrom(DelegatingLogStore.scala:127)
	at org.apache.spark.sql.delta.DeltaHistoryManager.getEarliestRecreatableCommit(DeltaHistoryManager.scala:367)
	at org.apache.spark.sql.delta.DeltaHistoryManager.checkVersionExists(DeltaHistoryManager.scala:327)
	at org.apache.spark.sql.delta.DeltaTableUtils$.resolveTimeTravelVersion(DeltaTable.scala:453)
	at org.apache.spark.sql.delta.catalog.DeltaTableV2.$anonfun$initialSnapshot$2(DeltaTableV2.scala:134)
	at scala.Option.map(Option.sca

In [ ]:
print("Clientes na versao atual:")
spark.sql("SELECT * FROM clientes").show()

In [ ]:
spark.stop()
print("Sessao Spark encerrada.")